# Nature-Style Disclosure Matrix Figure

Professional disclosure coverage matrix following Nature journal standards:
- Clean, minimal design
- Professional typography
- Subtle, effective color use
- High information density with clarity
- Publication-ready quality

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

WORKBOOK_NAME = "disclosure_coding.csv"

def find_plot_project():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "plots_for_paper"]
    for candidate in candidates:
        if (candidate / "data" / WORKBOOK_NAME).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate data/{WORKBOOK_NAME}")

PROJECT_DIR = find_plot_project()
DATA_FILE = PROJECT_DIR / "data" / WORKBOOK_NAME
PLOTS_DIR = PROJECT_DIR / "plots"
plot_data = pd.read_csv(DATA_FILE).sort_values("Plot order")

# Set Nature-style matplotlib parameters
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']
plt.rcParams['font.size'] = 8
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['xtick.major.size'] = 2
plt.rcParams['ytick.major.size'] = 2

print(f"? Loaded canonical plot data from {DATA_FILE}")


## Nature-Style Disclosure Matrix with Tier Grouping

In [ ]:
# Metrics and company order come from the canonical Plot_Data worksheet.
metrics_short = [
    "Boundary",
    "Total energy",
    "DC-only MWh",
    "Facility-level",
    "Region-resolved",
    "Emissions (Scope 2)",
    "Renewable %",
    "PUE",
    "Hourly CFE"
]
status_codes = {"No": 0, "Partial": 1, "Yes": 2}
value_labels = {value: key for key, value in status_codes.items()}

companies = plot_data["Company"].tolist()
df_status = plot_data.set_index("Company")[metrics_short]
unknown = set(df_status.stack().dropna().unique()) - set(status_codes)
if unknown:
    raise ValueError(f"Unknown disclosure status values in Plot_Data: {sorted(unknown)}")
df = df_status.replace(status_codes).astype(int)

print(f"? Data loaded: {len(companies)} companies ? {len(metrics_short)} metrics")


## Data Preparation

In [ ]:
# Define matrix dimensions
n_rows, n_cols = df.shape

# Nature-style color palette (professional, accessible)
colors_nature = {
    0: '#E0E0E0',  # Light gray - No
    1: '#90CAF9',  # Light blue - Partial
    2: '#2E7D32'   # Dark green - Yes
}

# Legend elements
legend_elements = [
    mpatches.Patch(facecolor=colors_nature[2], edgecolor='#1B5E20', label='Yes'),
    mpatches.Patch(facecolor=colors_nature[1], edgecolor='#42A5F5', label='Partial'),
    mpatches.Patch(facecolor=colors_nature[0], edgecolor='#BDBDBD', label='No')
]

# Tier structure for metrics
tier_info = [
    ("Accounting", 0, 1),
    ("Quantity", 1, 3),
    ("Location", 3, 5),
    ("Source/Accounting", 5, 8),
    ("Temporal", 8, 9),
]

fig, ax = plt.subplots(figsize=(7.5, 3.5), dpi=300)
fig.patch.set_facecolor('white')

# Add subtle alternating background bands per tier
tier_bg_colors = ['#F0F0F0', 'white']  # Light grey alternating with white for subtle tier grouping
for idx, (tier_name, start, end) in enumerate(tier_info):
    bg_rect = mpatches.Rectangle(
        (start, 0), end - start, n_rows,
        facecolor=tier_bg_colors[idx % 2],
        edgecolor='none',
        zorder=0
    )
    ax.add_patch(bg_rect)

# Add subtle emphasis for Hourly CFE column (highest resolution metric)
hourly_cfe_col = 8
emphasis_rect = mpatches.Rectangle(
    (hourly_cfe_col, 0), 1, n_rows,
    facecolor='#F7F9FC',  # Very subtle blue tint
    edgecolor='none',
    zorder=1
)
ax.add_patch(emphasis_rect)

# Draw cells
for i in range(n_rows):
    for j in range(n_cols):
        value = df.iloc[i, j]
        color = colors_nature[value]
        
        # Emphasize Hourly CFE column with thicker border
        if j == 8:  # Hourly CFE column
            edgecolor = '#666666'
            linewidth = 0.8
        else:
            edgecolor = '#DDDDDD'
            linewidth = 0.3
        
        rect = mpatches.Rectangle(
            (j, i), 1, 1,
            facecolor=color,
            edgecolor=edgecolor,
            linewidth=linewidth,
            zorder=2
        )
        ax.add_patch(rect)
        
        # Text
        text_color = 'white' if value == 2 else '#444444'
        ax.text(
            j + 0.5, i + 0.5,
            value_labels[value],
            ha='center',
            va='center',
            fontsize=7,
            color=text_color
        )

# Axes
ax.set_xlim(0, n_cols)
ax.set_ylim(-2, n_rows)  # Extended to include tier labels area
ax.invert_yaxis()

ax.set_xticks(np.arange(n_cols) + 0.5)
ax.set_yticks(np.arange(n_rows) + 0.5)
ax.set_xticklabels(metrics_short, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(companies, fontsize=8)

ax.tick_params(length=0, pad=3)

for spine in ax.spines.values():
    spine.set_visible(False)

# Tier separators - minimal and clean
for tier_name, start, end in tier_info:
    if start > 0:
        ax.axvline(x=start, color='#BBBBBB', linewidth=0.6, alpha=0.7, zorder=0)

# Tier labels above - all with consistent styling
y_label = -0.8
for tier_name, start, end in tier_info:
    x_center = (start + end) / 2
    
    # Add subtle background rectangle for tier label
    label_bg = mpatches.Rectangle(
        (start, y_label - 0.35), end - start, 0.7,
        facecolor='#F8F8F8',  # Very subtle light grey
        edgecolor='#E0E0E0',
        linewidth=0.3,
        zorder=1
    )
    ax.add_patch(label_bg)
    
    ax.text(
        x_center, y_label,
        tier_name,
        ha='center',
        va='top',
        fontsize=7,
        fontweight='normal',
        color='#555555',
        zorder=2
    )

# Legend
legend = ax.legend(
    handles=legend_elements,
    loc='upper left',
    bbox_to_anchor=(1.01, 1),
    frameon=False,
    fontsize=8,
    handlelength=1.2,
    handleheight=1.2
)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'disclosure_matrix_tiers_nature_style.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(PLOTS_DIR / 'disclosure_matrix_tiers_nature_style.png', dpi=300, bbox_inches='tight', facecolor='white')
print("✓ Saved: disclosure_matrix_tiers_nature_style.(pdf|png)")
plt.show()

---

## Summary

✓ **Nature-style figures created:**
- `disclosure_matrix_nature_style.(pdf|png)` - Clean, minimal matrix
- `disclosure_matrix_tiers_nature_style.(pdf|png)` - With tier structure

**Design principles applied:**
- Professional Arial/Helvetica typography
- Subtle, effective color palette
- Minimal line weights (0.5-0.8 pt)
- High contrast without being harsh  
- Clean white background
- Compact yet readable layout
- Publication-ready 300 DPI

**Key features:**
- Hourly CFE column emphasized with red border
- "Gold standard" label in italics
- Tier groupings with subtle separators
- Minimalist legend placement
- No unnecessary decorations

These match Nature journal figure standards for clarity and professionalism.